# Human in the Loop

Exploring the ability for HITL verification fo actions by the agent

In [1]:
from langchain.agents import create_agent, AgentState
from langchain.messages import HumanMessage
from langchain.tools import tool, ToolRuntime
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv

from pprint import pprint

In [2]:
load_dotenv()

True

In [3]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an meail from the given address"""
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body"""
    return f"Email sent"

Setting up the agent state and creating the agent with the HITL middleware. This middleware specifies which tool calls require human input before executing.

In [4]:
# Setting up the Agent's State
class EmailState(AgentState):
    email: str

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware[AgentState, None](
            interrupt_on={
                "read_email": False,
                "send_email": True
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [5]:
config = {"configurable": {"thread_id": "1"}}

### Approving

Approving the HITL interrupted action reqiures invoking the agent with a Command object specifying the decision. In this case it receives a tuple with a named dict called "resume" and the included decisions for the agent. This requires memory and passing the same config thread.

In [6]:
approval = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config
)

In [7]:
pprint(approval)

{'email': 'Hi Sam, can I add a topic to the agenda for our meeting tomorrow?',
 'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='d19c887e-3ab3-4f3d-8251-00c93f102369'),
              AIMessage(content=[{'text': "I'll read your email first, and then I can help you send a response.", 'type': 'text'}, {'id': 'toolu_012czbaPmmVmqRuK6zqAfTHJ', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeapGavbUMe3SY5cWpwzk', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 614, 'output_tokens': 54, 'output_tokens_details': None, 'server_tool_use': Non

In [8]:
response = agent.invoke(
    Command[tuple[()]](
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

pprint(response)

{'email': 'Hi Sam, can I add a topic to the agenda for our meeting tomorrow?',
 'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='d19c887e-3ab3-4f3d-8251-00c93f102369'),
              AIMessage(content=[{'text': "I'll read your email first, and then I can help you send a response.", 'type': 'text'}, {'id': 'toolu_012czbaPmmVmqRuK6zqAfTHJ', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeapGavbUMe3SY5cWpwzk', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 614, 'output_tokens': 54, 'output_tokens_details': None, 'server_tool_use': Non

### Rejecting

Rejecting the tool call is very similar -- pass a command with a reject decision. This is clunky as the agent will request another approval of the tool call before executing.

In [9]:
config2 = {"configurable": {"thread_id": "2"}}

reject = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config2
)

In [10]:
reject

{'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='0bd5755d-80a7-47d9-97b0-c919ec26caa2'),
  AIMessage(content=[{'text': "I'll read your email first to see what needs a response.", 'type': 'text'}, {'id': 'toolu_01Au5XyfP9fkveYXeVKfQq8j', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeapLSFKRwiot8WwbJKqx', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 614, 'output_tokens': 50, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic

In [11]:
reject = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": "Don't send this reply. Draft a response that asks the sender to pick a specific topic."
                }
            ]
        }
    ),
    config=config2
)

In [12]:
reject

{'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='0bd5755d-80a7-47d9-97b0-c919ec26caa2'),
  AIMessage(content=[{'text': "I'll read your email first to see what needs a response.", 'type': 'text'}, {'id': 'toolu_01Au5XyfP9fkveYXeVKfQq8j', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeapLSFKRwiot8WwbJKqx', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 614, 'output_tokens': 50, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 'model_provider': 'anthropic

### Editing

Editing removes the clunkiness introduced by rejecting -- it changes the behavior of the agent and lets it proceed with execution without a second approval.

In [12]:
config3 = {"configurable": {"thread_id": "3"}}

edit = agent.invoke(
    {
        "messages":[HumanMessage(content="Please read my email and send a response")],
        "email": "Hi Sam, can I add a topic to the agenda for our meeting tomorrow?",
    },
    config=config3
)

In [13]:
edit = agent.invoke(
    Command[tuple[()]](
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {"body": "Yes, let's discuss the new project on agent workflows."}
                    }
                }
            ]
        }
    ),
    config=config3
)

In [14]:
edit

{'messages': [HumanMessage(content='Please read my email and send a response', additional_kwargs={}, response_metadata={}, id='19b8e693-9c31-4ed2-b2ed-e8a0300114ae'),
  AIMessage(content=[{'text': "I'll read your email first to see what it says, then I can help you send a response.", 'type': 'text'}, {'id': 'toolu_016Pk9C51SGyv3t9GFnPztNb', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'read_email', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeaovhvDznv3UGwxvfH4W', 'container': None, 'model': 'claude-haiku-4-5-20251001', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'not_available', 'input_tokens': 614, 'output_tokens': 58, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-haiku-4-5-20251001', 